In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# Create dataset
X = []
y = []

for i in range(100):
    X.append([i, i+1, i+2])
    y.append(i+3)

X = np.array(X)
y = np.array(y)

# Reshape for RNN: (samples, timesteps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))

print("Input shape:", X.shape)

model = Sequential([
    SimpleRNN(32, activation='tanh', input_shape=(3, 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

model.summary()

model.fit(X, y, epochs=20, verbose=1)

test_input = np.array([[3, 4, 5]])
test_input = test_input.reshape((1, 3, 1))

prediction = model.predict(test_input)
print("Prediction:", prediction)

Input shape: (100, 3, 1)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 3578.9968
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3554.8503 
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3533.6450
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3512.2642 
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3490.0640 
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3465.2002 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3437.5012 
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3411.8010
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3378.3782 
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3344.3088 
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3314.3882 
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3294.0007 
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3277.8091 
Epoch 14/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3261.9563 
Epoch 15/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding

# Sample text
text = "deep learning is powerful and deep learning is fun"

# Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

sequences = []
words = text.split()

for i in range(1, len(words)):
    seq = words[:i+1]
    sequences.append(tokenizer.texts_to_sequences([seq])[0])

# Pad sequences
max_len = max(len(seq) for seq in sequences)
sequences = pad_sequences(sequences, maxlen=max_len)

# Split into X and y
X = sequences[:, :-1]
y = sequences[:, -1]

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)

model = Sequential([
    Embedding(vocab_size, 10, input_length=max_len-1),
    SimpleRNN(32),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

model.fit(X, y, epochs=100, verbose=0)

def predict_next_word(seed_text):
    seq = tokenizer.texts_to_sequences([seed_text])[0]
    seq = pad_sequences([seq], maxlen=max_len-1)

    pred = model.predict(seq, verbose=0)
    predicted_word = tokenizer.index_word[np.argmax(pred)]

    return predicted_word

print(predict_next_word("deep learning is"))

Vocabulary size: 7
powerful
